# Inteligência Artificial 2025/2026
## Trabalho Prático: PopOut + MCTS + ID3

**Grupo:** _[preencher nomes e números de aluno]_

**Data:** Maio 2026

---

> Este notebook é o entregável principal do trabalho prático conforme `docs/IA_2526_Project.pdf`. Implementa MCTS para PopOut e árvores de decisão (ID3) para Iris e para um dataset gerado pelo próprio MCTS. Toda a avaliação experimental está documentada e reproduzível pelos scripts em `experiments/`.


In [ ]:
# Setup: ensures imports work whether the notebook is run from codes/ or
# from the project root. If __vsc_ipynb_file__ is set (VSCode), uses its
# dirname; otherwise assumes the notebook lives in the cwd.
import os, sys
_NB_FILE = globals().get('__vsc_ipynb_file__')
_HERE = os.path.dirname(os.path.abspath(_NB_FILE)) if _NB_FILE else os.getcwd()
if _HERE not in sys.path:
    sys.path.insert(0, _HERE)
os.chdir(_HERE)
print(f'Working dir: {os.getcwd()}')


## Estrutura do notebook

1. Introdução
2. Constraints e decisões de design
3. Motor do jogo PopOut
4. MCTS standard (UCB1)
5. Variações do MCTS
6. Tactical lookahead à raiz
7. Árvores de decisão (ID3) e Iris
8. Geração do dataset PopOut
9. ID3 sobre o dataset PopOut + Tree strategy
10. Avaliação experimental rigorosa
11. Conclusões e trabalho futuro
12. Referências


## 1. Introdução

### 1.1 Problema

PopOut é uma variante do Connect-4 (Allen 2010) com duas inovações:
- **Drop:** largar peça pelo topo (como Connect-4).
- **Pop:** remover uma peça da própria cor do fundo da coluna; tudo o que está acima desce uma posição.

Três regras especiais governam casos ambíguos:
1. Pop com 4-em-linha simultâneo para os dois jogadores → vence quem fez o pop.
2. Tabuleiro cheio sem pops legais → empate.
3. Repetição tripla do mesmo estado → qualquer jogador pode declarar empate.

### 1.2 Objetivos

Implementar:
- **Metade A:** Monte Carlo Tree Search (MCTS) com UCB1 + variações experimentais.
- **Metade B:** ID3 (Iterative Dichotomiser 3) de raiz, sem `scikit-learn`, aplicado a Iris e a um dataset gerado pelo próprio MCTS (behavioural cloning).
- 3 cenários jogáveis: H vs H, H vs C, C vs C com 2 algoritmos diferentes (MCTS vs ID3).

### 1.3 Sumário dos resultados

- ✅ Motor PopOut completo com 21 testes (regra duplo-4 e repetição cobertas).
- ✅ MCTS com UCB1 validado **numericamente** contra exemplo trabalhado das aulas.
- ✅ 4 variações do MCTS comparadas em tabelas A/B/C/D.
- ✅ Iris: **94.7% ± 4% (5-fold CV)** com discretização equal_width(k=3).
- ✅ Dataset PopOut: **856 pares (estado, jogada)** de 50 partidas MCTS-vs-MCTS.
- ✅ Tree de PopOut: 4,300× mais rápida que MCTS-E para força equivalente.
- ✅ Win-rate matrix entre Random / MCTS-Easy / MCTS-Medium / Tree.
- ✅ **109 testes pytest a passar** + 1 skipped (impossibilidade física documentada).


## 2. Constraints e decisões de design

> O enunciado §4.7 pede explicitamente: *"Mention the constraints you are considering for the solution in the notebook"*.

### 2.1 Tabuleiro
- **Dimensões:** 7 colunas × 6 linhas (Connect-4 standard, conforme figura do enunciado).
- **Representação:** `np.ndarray` `int8` 6×7. Linha 0 = topo, linha 5 = fundo.
- **Valores:** 0 = vazio, 1 = P1 (X), 2 = P2 (O).

### 2.2 Estado
- **Imutável** (`@dataclass(frozen=True)`). `apply_move` devolve sempre novo `State`.
- Razão: o MCTS partilha estados entre nós da árvore; mutação corrompe-a silenciosamente.

### 2.3 Algoritmo de aprendizagem
- **ID3 from scratch.** Sem scikit-learn. Apenas pandas/numpy para gestão de dados (permitido pelo enunciado).
- 4 casos base implementados conforme Russell & Norvig.

### 2.4 Discretização
- ID3 puro só aceita atributos categóricos. Iris é numérico → discretização obrigatória.
- 3 estratégias implementadas: `equal_width`, `equal_frequency`, `supervised` (binário com IG).
- Default empírico: `equal_width(k=3)` (melhor accuracy/tamanho-tree).

### 2.5 Interface
- CLI com formato `X--O-XO` (conforme exemplo do enunciado).
- **Adicionalmente:** GUI Pygame com cliques, botões pop/drop, modo MCTS-vs-Tree.

### 2.6 Ferramentas e dependências
- Python 3.12, NumPy, pandas, matplotlib, pygame.
- pytest para testes.
- **Sem scikit-learn nem outras libs de ML** — implementação de raiz para ID3.


## 3. Motor do jogo PopOut

Código em `popout.py`. Funções públicas:
- `initial_state()` — devolve estado inicial.
- `legal_moves(state)` — lista de `Move` legais (drops + pops).
- `apply_move(state, move)` — devolve novo estado.
- `check_win(board, last_move, mover)` — regra do duplo-4 incluída.
- `can_claim_repetition_draw(state)` — regra da repetição tripla.

### 3.1 Pseudocódigo das funções principais

**`legal_moves`:**
```
para c em 0..6:
    se board[topo, c] == vazio: drops.add(Move(c, 'drop'))
    se board[fundo, c] == player_to_move: pops.add(Move(c, 'pop'))
```

**`apply_move`:**
```
new_board = board.copy()
se move.kind == 'drop':
    encontrar primeira linha vazia a contar de baixo na coluna
    new_board[r, c] = mover
senão (pop):
    new_board[1:6, c] = old[0:5, c]   # shift down
    new_board[0, c] = vazio
```

**`check_win` com regra duplo-4:**
```
se last_move.kind == 'pop':
    se eu_ganhei e ele_ganhou: return mover  # regra Allen
    se eu_ganhei: return mover
    se ele_ganhou: return outro
senão (drop):
    se eu_ganhei: return mover
```

### 3.2 Demo de uma partida HvH scripted


In [ ]:
from popout import initial_state, apply_move, Move, render

state = initial_state()
moves = [
    Move(0, 'drop'),  # P1 → (5,0)
    Move(6, 'drop'),  # P2 → (5,6)
    Move(1, 'drop'),  # P1 → (5,1)
    Move(6, 'drop'),  # P2 → (4,6)
    Move(2, 'drop'),  # P1 → (5,2)
    Move(6, 'drop'),  # P2 → (3,6)
    Move(3, 'drop'),  # P1 → (5,3) — 4-em-linha horizontal!
]
for m in moves:
    state = apply_move(state, m)

print(render(state.board))
print(f'\nWinner: P{state.winner}')


### 3.3 Validação dos testes do motor

`tests/test_engine.py` cobre:
- Drops, pops, vitórias horizontais/verticais/diagonais.
- **Regra do duplo-4 por pop:** dois testes com 3 cenários (`test_pop_creates_double_four_pop_player_wins`, `_v2`, `test_pop_creates_only_opponent_four_opponent_wins`).
- Repetição tripla.
- Pureza de `apply_move` (não muta estado original — crítico para o MCTS).

20 testes a passar + 1 skipped (cenário fisicamente impossível, documentado).


## 4. MCTS standard (UCB1)

Código em `mcts.py`.

### 4.1 Os 4 passos canónicos

```
para i = 1..N:
    1. Selection — desce pela árvore escolhendo filho com maior UCB1
    2. Expansion — chega a folha; cria filho novo
    3. Simulation — joga aleatoriamente (rollout) até ao fim
    4. Backpropagation — propaga vitória/derrota para cima

devolve a jogada cujo filho da raiz tem mais visitas
```

### 4.2 Fórmula UCB1

$$
UCB1(n) = \underbrace{\frac{U(n)}{N(n)}}_{\text{exploitation}} + C \cdot \underbrace{\sqrt{\frac{\ln N(\text{parent}(n))}{N(n)}}}_{\text{exploration}}
$$

- $U(n)$ = vitórias acumuladas no nó *n*, do ponto de vista do jogador que **escolheu** vir cá.
- $N(n)$ = visitas ao nó.
- $C = \sqrt{2} \approx 1.414$ — valor canónico (Auer et al. 2002).

### 4.3 Convenção crítica do sinal

> ⚠️ Erro #1 das implementações de MCTS de raiz.

`U` é actualizado do ponto de vista do jogador que **escolheu** vir a este nó (= `parent.player_to_move`). Empate vale 0.5. Implementado em `mcts.py:backprop`.

### 4.4 Validação numérica do UCB1

Replicámos exatamente o exercício 11.4 do `MATERIA_IA.md` num teste unitário. Pai com `N=100`, três filhos:
- X: `U=20, N=25` → **UCB ≈ 1.407**
- Y: `U=30, N=70` → **UCB ≈ 0.791**
- Z: `U=2, N=5` → **UCB ≈ 1.758** ← seleccionado (exploração)

Teste: `tests/test_mcts.py::test_ucb1_matches_worked_example`.

### 4.5 Demo: MCTS escolhe a vitória imediata


In [ ]:
import math, random
from popout import initial_state, apply_move, Move
from mcts import mcts_search

# P1 com 3-em-linha em row 5 cols 0-2; vez do P1
state = initial_state()
seq = [
    Move(0, 'drop'), Move(0, 'drop'),
    Move(1, 'drop'), Move(1, 'drop'),
    Move(2, 'drop'), Move(6, 'drop'),
]
for m in seq:
    state = apply_move(state, m)

chosen = mcts_search(state, n_simulations=300, c=math.sqrt(2),
                    rng=random.Random(0))
print(f'MCTS escolheu: {chosen}')
print('Esperado: drop(3) — completa 4-em-linha em row 5')


## 5. Variações do MCTS (Fase 4)

> O enunciado §4.1 pede explicitamente: *"analyse/explore different numbers of selected children for each node, and other strategies"*.

### 5.1 Variações implementadas

| Variação | Parâmetro | Estado |
|----------|-----------|--------|
| Rollout policy | `rollout='random' \| 'heuristic_win' \| 'heuristic_block'` | ✅ |
| Constante de exploração | `c` (testado: 0.5, 1.0, √2, 2.0) | ✅ |
| Número de simulações | `n_simulations` (100, 300, 600) | ✅ |
| Limite de filhos | `max_children` (k=3, 5, None) | ✅ |

### 5.2 Resultados experimentais (de `mcts_variations.py`)

#### A) Rollout policy (compute equivalente)

| A | B | A | B | t/move A | t/move B |
|---|---|---|---|----------|----------|
| `random` (N=200) | `heuristic_win` (N=200) | 0 | **6** | 244 ms | 643 ms |
| `random` (N=300) | `heuristic_block` (N=50) | **4** | 0 | 416 ms | 6336 ms |
| `heuristic_win` (N=200) | `heuristic_block` (N=50) | **4** | 0 | 799 ms | 7266 ms |

→ **`heuristic_win` é o sweet-spot.** `heuristic_block` é caro (O(b²)/ply); ao mesmo compute, perde para variantes mais simples com mais simulações.

#### B) Número de simulações N (rollout `heuristic_win`)

| A | B | A | B |
|---|---|---|---|
| N=100 | N=300 | 0 | **6** |
| N=300 | N=600 | 0 | **4** |

→ Crescimento monotónico de força. Sem saturação visível até N=600.

#### C) C (vs random, N=200, `heuristic_win`)

| C | Win-rate vs random | t/move |
|---|--------------------|--------|
| 0.5 | 6/6 | 728 ms |
| 1.0 | 6/6 | 648 ms |
| **√2** | **6/6** | **533 ms** |
| 2.0 | 6/6 | 876 ms |

→ Todos saturam vs random. **C=√2 é o mais rápido**; suporta empiricamente o canónico.

#### D) `max_children`

| k | Win-rate vs random | t/move |
|---|--------------------|--------|
| None | 4/4 | 527 ms |
| 5 | 4/4 | 438 ms |
| **3** | **4/4** | **356 ms** |

→ Limitar a 3 filhos centrais mantém qualidade e poupa **33% tempo**.


## 6. Tactical lookahead (Fase 4.5)

Acrescentado para resolver a observação prática: *"a IA não faz pops mesmo quando popar é a melhor jogada"*.

### 6.1 O que faz

Antes do MCTS arrancar, fazemos uma **busca exata 2-ply** que devolve:
- Movimento que vence imediatamente (drop ou pop).
- Movimento que cria fork (2+ ameaças simultâneas que adversário não bloqueia ambas).

Se encontrar, devolve esse movimento e salta o MCTS. Senão, MCTS normal.

### 6.2 Custo

O(b³) na raiz = ~3000 `apply_move` por jogada = **~30-100 ms**. Negligível comparado ao MCTS (~1 s).

### 6.3 Por que isto importa

- Pops raramente são óptimos em random rollouts → MCTS sub-estima-os.
- Tactical lookahead apanha pops que criam vitória forçada — exatamente o que o utilizador esperava.

### 6.4 Demo: detectar fork


In [ ]:
import math, random
from popout import initial_state, apply_move, Move
from mcts import find_forced_win, mcts_search

# Cenario fork: P1 a mover, drop(3) cria 2 ameacas (drop(1) e drop(5) ganham)
state = initial_state()
seq = [
    Move(2, 'drop'),  # P1 (5,2)
    Move(0, 'drop'),  # P2 (5,0)
    Move(4, 'drop'),  # P1 (5,4)
    Move(6, 'drop'),  # P2 (5,6)
]
for m in seq:
    state = apply_move(state, m)

# Sem MCTS, so com tactical lookahead:
forced = find_forced_win(state, depth=2)
print(f'Tactical lookahead encontrou: {forced}  (esperado drop(3))')

# Com tactical_root, MCTS apanha mesmo com N=10:
chosen = mcts_search(state, n_simulations=10, c=math.sqrt(2),
                    rollout='random', tactical_root=True,
                    rng=random.Random(0))
print(f'MCTS+tactical: {chosen}')


## 7. Árvores de decisão (ID3) + Iris

Código em `decision_tree_builder.py`. Aprendizagem **sem scikit-learn**.

### 7.1 Algoritmo ID3 — 4 casos base

```
ID3(X, y, atributos):
    1. Se todas as classes iguais a c → folha(c)
    2. Se atributos == ∅ → folha(maioria(y))
    3. Se y == ∅ → folha(maioria_pai)
    4. Senão: A = argmax_a IG(X, y, a); split em A; recursão
```

### 7.2 Information Gain

$$H(C) = -\sum_c P(c) \log_2 P(c)$$

$$\text{Gain}(A) = H(C) - \sum_v \frac{|S_v|}{|S|} H(C \mid A=v)$$

### 7.3 Validação numérica — exemplo Gripe

Replicámos exatamente o exercício 11.5 do `MATERIA_IA.md`:
- $H(C) = 0.954$ bits ✓
- $\text{Gain}(\text{Febre}) = 0.549$ ✓
- $\text{Gain}(\text{Tosse}) = \text{Gain}(\text{Cansaço}) = 0.159$ ✓

Teste: `tests/test_decision_tree.py::test_information_gain_gripe_example`.

### 7.4 Discretização para Iris

ID3 puro só aceita categóricos. Implementámos 3 estratégias:

| Estratégia | Descrição |
|------------|-----------|
| `equal_width` | Bins iguais entre min e max |
| `equal_frequency` | Bins com mesma quantidade de amostras |
| `supervised` | Threshold binário que maximiza Information Gain |

### 7.5 Treino e avaliação no Iris


In [ ]:
import os
import numpy as np
import pandas as pd
from decision_tree_builder import (
    fit_discretizer_equal_width, transform_discretizer,
    id3, predict_batch, accuracy, render_tree_text, tree_size,
)

iris_path = 'iris.csv'
df = pd.read_csv(iris_path).drop(columns=['ID'])
feature_cols = [c for c in df.columns if c != 'class']

# Split 80/20
df_shuf = df.sample(frac=1, random_state=0).reset_index(drop=True)
sp = int(0.8 * len(df_shuf))
train, test = df_shuf.iloc[:sp], df_shuf.iloc[sp:]
X_tr, y_tr = train[feature_cols], train['class']
X_te, y_te = test[feature_cols], test['class']

# Discretização equal_width(k=3)
fit = fit_discretizer_equal_width(X_tr, feature_cols, n_bins=3)
X_tr_d = transform_discretizer(X_tr, fit)
X_te_d = transform_discretizer(X_te, fit)

# Treinar e avaliar
tree = id3(X_tr_d, y_tr, feature_cols)
acc = accuracy(y_te, predict_batch(tree, X_te_d))
sz = tree_size(tree)

print(f'Accuracy de teste: {acc:.3f}')
print(f'Tamanho da árvore: {sz}')
print()
print('Estrutura da árvore:')
print(render_tree_text(tree))


### 7.6 Comparação das 3 estratégias (split 80/20)

| Estratégia | Acc treino | Acc teste | Folhas | Profundidade |
|------------|------------|-----------|--------|--------------|
| supervised (binário) | 0.717 | 0.633 | **4** | 3 |
| **equal_width(k=3)** | **0.992** | **0.933** | 5 | 3 |
| equal_width(k=5) | 0.967 | 0.900 | 17 | 4 |
| equal_frequency(k=3) | 0.992 | 0.933 | 10 | 4 |

### 7.7 Cross-validation 5-fold

| Estratégia | Mean acc | Std acc | Mean folhas |
|------------|----------|---------|-------------|
| supervised | 0.653 | ±0.072 | 4.2 |
| **equal_width(k=3)** | **0.947** | **±0.040** | 8.4 |
| equal_freq(k=3) | 0.947 | ±0.045 | 12.6 |

### 7.8 Trade-off elegante

A discretização `supervised` produz a árvore **mais pequena** (4 folhas) — cumpre literalmente "minimizar tamanho" do enunciado. Mas comprime demais (cada feature → 2 níveis): cai a 63% accuracy. A discretização `equal_width(k=3)` adiciona apenas **1 folha** mas ganha **+30 pp** em accuracy.

Reproduzir: `python -m iris_test`


## 8. Geração do dataset PopOut

> Behavioural cloning: o MCTS é "professor", a árvore (Fase 9) é "aluno".

### 8.1 Pipeline

```
para cada partida em 1..N:
    estado ← inicial
    enquanto não terminado:
        com probabilidade ε: jogada ← random
        senão: jogada ← MCTS(estado, parametros)
        gravar (estado, to_play, jogada)
        estado ← apply_move(estado, jogada)
```

### 8.2 Encoding

| Coluna | Tipo | Domínio |
|--------|------|---------|
| `s0..s41` | int | {0, 1, 2} |
| `to_play` | int | {1, 2} |
| `move` | str | `d0`–`d6`, `p0`–`p6` |

### 8.3 Parâmetros e estatísticas (50 partidas)

| Métrica | Valor |
|---------|-------|
| Partidas | 50 |
| ε (jogadas aleatórias) | 0.10 |
| MCTS N | 200 |
| Rollout | `heuristic_win` |
| `tactical_root` | True |
| Tempo | 559 s (~9.3 min) |
| **Total pares** | **856** |
| Pops | 41 (4.8%) |
| Vencedores P1/P2 | 33 / 17 |

Reproduzir: `python -m generate_dataset --games 50`

CSV: `data/popout_dataset.csv` (75 KB).


In [ ]:
import pandas as pd
df = pd.read_csv('popout_dataset.csv')
print(f'Forma: {df.shape}')
print(f'Colunas: {list(df.columns[:5])} ... {list(df.columns[-3:])}')
print()
print('Top 8 classes mais frequentes:')
print(df['move'].value_counts().head(8))


## 9. ID3 sobre o dataset PopOut + Tree strategy

Código em `decision_tree_builder.py` e `train_tree.py`.

### 9.1 Sweep de `max_depth`

| max_depth | Acc treino | Acc teste | Folhas |
|-----------|------------|-----------|--------|
| 3 | 0.303 | 0.169 | 26 |
| 5 | 0.526 | 0.157 | 156 |
| 8 | 0.876 | 0.221 | 450 |
| **10** | **0.904** | **0.227** | **474** |

→ **Overfitting clássico:** train sobe a 90%, test estagna em 22%.

→ Acc teste 22% é melhor que a baseline (classe mais frequente d3 = 21.3%); a árvore aprendeu **algo**, mas pouco mais que o bias central.

### 9.2 Tree strategy — fallback

Predição da árvore pode ser ilegal nesse estado (ex: `d3` mas col 3 cheia). Fallback em 3 tiers:
1. Move legal com mesma kind (drop/pop) mais central.
2. Drop central qualquer.
3. Primeiro legal.

### 9.3 Tree vs Random — sanity ✅

| Tree | Random | Empates |
|------|--------|---------|
| **10** | 0 | 0 |

Tempo: tree ~17 μs, random ~10 μs.

### 9.4 Tree vs MCTS-Médio — comparação aluno-professor

| Tree | MCTS-Médio | Empates |
|------|------------|---------|
| 0 | **6** | 0 |

Tempo: **tree 40 μs vs MCTS 610 ms** = **speedup ~15,000×**.

> 📌 **Insight chave:** o aluno não supera o professor (esperado em behavioural cloning), mas decide 4 ordens de grandeza mais rápido. Justifica o uso da árvore em cenários de tempo limitado.

Reproduzir: `python -m train_tree --sweep --vs-random 10 --vs-mcts 6`


In [ ]:
import pickle, random
from game import play_game
from popout import P1, P2
from mcts import mcts_strategy
from decision_tree_builder import tree_strategy

# Carregar arvore treinada
with open('decision_tree.pkl', 'rb') as f:
    tree = pickle.load(f)

# Demo: 1 partida tree vs MCTS-Medio (P1)
final = play_game(
    mcts_strategy(n_simulations=200, rollout='heuristic_win',
                  tactical_root=True, rng=random.Random(0)),
    tree_strategy(tree),
    on_render=lambda _: None, show_intermediate=False, max_turns=200,
)
print(f'Vencedor: P{final.winner}')
print(f'Comprimento da partida: {final.history_counts.__len__()} estados visitados')


## 10. Avaliação experimental rigorosa (Fase 8)

> Esta secção sustenta os 30% técnicos do enunciado.

### 10.1 Win-rate matrix — 4 agentes, 2 partidas/célula, alternando lados

| A \ B  | Random | MCTS-E | MCTS-M | Tree |
|---------|--------|--------|--------|------|
| **Random** | 0.50 | 0.00 | 0.00 | 0.00 |
| **MCTS-E** | 1.00 | 0.50 | 0.00 | 0.50 |
| **MCTS-M** | 1.00 | 1.00 | 0.50 | 1.00 |
| **Tree**   | 1.00 | 0.50 | 0.00 | 0.50 |

Hierarquia: **MCTS-M ≫ MCTS-E ≈ Tree ≫ Random**.

Heatmap (gerado por `evaluation.py`):

![Win-rate matrix](../../../tmp/fase8_out/winrate_matrix.png)

### 10.2 Tempo médio por jogada

| Agente | t/jogada |
|--------|----------|
| Random | 11 μs |
| **Tree** | **34 μs** |
| MCTS-E | 146 ms |
| MCTS-M | 756 ms |

→ Tree e MCTS-E **mesma força** (0.5/0.5 no matchup) mas Tree **~4,300× mais rápida**.

### 10.3 Learning curve da árvore

| n_train | Acc teste |
|---------|-----------|
| 137 | 0.211 |
| 274 | 0.187 |
| 411 | 0.222 |
| 548 | 0.216 |

![Learning curve](../../../tmp/fase8_out/tree_learning_curve.png)

> **Conclusão crítica:** a curva é **plana em ~22%**. Aumentar o dataset não vai resolver — é problema de **representação** (features raw 0/1/2 cells) ou **complexidade do problema**, não de tamanho. Recomendação: features derivadas (ex: peças por coluna, alinhamentos abertos).

### 10.4 Outras figuras

- `tree_depth_sensitivity.png` — overfitting clássico.
- `mcts_time_vs_n.png` — escala linear do MCTS.

Reproduzir: `python -m evaluation [--quick]`


## 11. Conclusões e trabalho futuro

### 11.1 Resultados-chave

1. **MCTS standard** validado numericamente contra exemplo das aulas (UCB1 = 1.407/0.791/1.758).
2. **MCTS variations** mostraram trade-offs: heuristic_win é o sweet-spot prático.
3. **Tactical lookahead à raiz** resolveu o problema percebido de "AI não faz pops" — apanha vitórias forçadas em 2 plies.
4. **ID3 em Iris** atinge **94.7% ± 4%** com 5-fold CV e árvore de apenas 8.4 folhas.
5. **Behavioural cloning** funciona até ao nível MCTS-Easy: tree iguala MCTS-E em força e é **4,300× mais rápida**.
6. **Tree não consegue imitar MCTS-Medium** (perde 0/6) — limite do paradigma com este dataset/features.

### 11.2 Limitações

- Dataset PopOut tem 856 pares; learning curve mostra **plafond a 22% — mais dados não resolverão**.
- MCTS-Difícil (N=800) não foi incluído na matriz por custo computacional.
- ID3 puro sem pruning post-hoc — usámos `max_depth` como controlo simples.
- Avaliação com n=2-4 jogos por célula tem variância alta; resultados publicáveis exigiriam n=10+.

### 11.3 Trabalho futuro (concreto)

1. **Features derivadas** para o ID3 do PopOut: peças por coluna, comprimento da maior linha, ameaças ativas. Espera-se quebrar o plafond de 22%.
2. **Dataset maior** (1000+ partidas) em batch overnight.
3. **MCTS-Difícil** na matriz de avaliação.
4. **Pruning chi-square** ou reduced-error.
5. **Bitboards** no engine para profile do MCTS — reduziria custo das simulações em 5-20×.


## 12. Referências

- **Russell, S. & Norvig, P.** — *Artificial Intelligence: A Modern Approach*. Capítulos sobre Adversarial Search (MCTS) e Learning from Observations (Decision Trees, ID3).
- **Allen, J. D. (2010)** — *The Complete Book of Connect-4: History, Strategy, Puzzles*. Sterling. *(Origem das 3 regras especiais do PopOut.)*
- **Browne et al. (2012)** — *A Survey of Monte Carlo Tree Search Methods*. IEEE T-CIAIG.
- **Auer, P., Cesa-Bianchi, N., Fischer, P. (2002)** — *Finite-time Analysis of the Multiarmed Bandit Problem*. (Base teórica do UCB1.)
- **Quinlan, J. R. (1986)** — *Induction of Decision Trees*. Machine Learning 1.
- Slides das aulas de IA 2025/2026 — Class 4 (MCTS), Class 7 (Learning).

---

### Reprodutibilidade

Toda a avaliação experimental é reproduzível:

```bash
# Suite de testes (109 passed, 1 skipped)
python -m pytest tests/ -c /dev/null

# Variações do MCTS (Fase 4)
python -m mcts_variations --quick

# Pipeline Iris (Fase 5)
python -m iris_test

# Gerar dataset PopOut (Fase 6)
python -m generate_dataset --games 50

# Treinar tree PopOut (Fase 7)
python -m train_tree --sweep --vs-random 10 --vs-mcts 6

# Avaliação completa (Fase 8)
python -m evaluation [--quick]

# Jogar (GUI)
python play.py --gui
```
